## 1.Load Data

In [2]:
import pandas as pd

In [23]:
df = pd.read_csv('RD202401.csv', sep= ';', encoding='latin-1', low_memory=False)

## 2.Know the Basic structure: shape, columns, duplicates

In [ ]:
print(df.head())

In [25]:
df.shape
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1107877 entries, 0 to 1107876
Columns: 113 entries, UF_ZI to TPDISEC9
dtypes: float64(28), int64(73), str(12)
memory usage: 955.1 MB


np.int64(0)

In [ ]:
df.isnull().sum().sort_values(ascending=False)


In [27]:
df.dtypes.value_counts()

int64      73
float64    28
str        12
Name: count, dtype: int64

In [28]:
list(df.columns)

['UF_ZI',
 'ANO_CMPT',
 'MES_CMPT',
 'ESPEC',
 'CGC_HOSP',
 'N_AIH',
 'IDENT',
 'CEP',
 'MUNIC_RES',
 'NASC',
 'SEXO',
 'UTI_MES_IN',
 'UTI_MES_AN',
 'UTI_MES_AL',
 'UTI_MES_TO',
 'MARCA_UTI',
 'UTI_INT_IN',
 'UTI_INT_AN',
 'UTI_INT_AL',
 'UTI_INT_TO',
 'DIAR_ACOM',
 'QT_DIARIAS',
 'PROC_SOLIC',
 'PROC_REA',
 'VAL_SH',
 'VAL_SP',
 'VAL_SADT',
 'VAL_RN',
 'VAL_ACOMP',
 'VAL_ORTP',
 'VAL_SANGUE',
 'VAL_SADTSR',
 'VAL_TRANSP',
 'VAL_OBSANG',
 'VAL_PED1AC',
 'VAL_TOT',
 'VAL_UTI',
 'US_TOT',
 'DT_INTER',
 'DT_SAIDA',
 'DIAG_PRINC',
 'DIAG_SECUN',
 'COBRANCA',
 'NATUREZA',
 'NAT_JUR',
 'GESTAO',
 'RUBRICA',
 'IND_VDRL',
 'MUNIC_MOV',
 'COD_IDADE',
 'IDADE',
 'DIAS_PERM',
 'MORTE',
 'NACIONAL',
 'NUM_PROC',
 'CAR_INT',
 'TOT_PT_SP',
 'CPF_AUT',
 'HOMONIMO',
 'NUM_FILHOS',
 'INSTRU',
 'CID_NOTIF',
 'CONTRACEP1',
 'CONTRACEP2',
 'GESTRISCO',
 'INSC_PN',
 'SEQ_AIH5',
 'CBOR',
 'CNAER',
 'VINCPREV',
 'GESTOR_COD',
 'GESTOR_TP',
 'GESTOR_CPF',
 'GESTOR_DT',
 'CNES',
 'CNPJ_MANT',
 'INFEHOSP',
 'C

Load the dictionary file
check column names

In [3]:
dict_df = pd.read_csv('dict_SIH.csv', sep= ',', encoding='latin-1')

In [30]:
print(dict_df.columns.tolist())

['id', 'type', 'description']


Load dataset and inspect columns

In [4]:
rd01_df=pd.read_csv('RD202401.csv', sep= ';', encoding='latin-1', low_memory=False) 

#Get the list of columns in RD202401.csv
rd_columns = rd01_df.columns.tolist()
print(len(rd_columns),"columns")
print(rd_columns[:10]) #show first 10 columns for quick look    

113 columns
['UF_ZI', 'ANO_CMPT', 'MES_CMPT', 'ESPEC', 'CGC_HOSP', 'N_AIH', 'IDENT', 'CEP', 'MUNIC_RES', 'NASC']


Compare dictionary ids vs dataset columns

In [5]:
dict_ids =set(dict_df['id'].unique())
rd_columns_set = set(rd_columns)
matching_vars = rd_columns_set.intersection(dict_ids)
print(f"matching_vars: {len(matching_vars)}")
print(sorted(matching_vars)[:20]) #show first 10 matching variables

matching_vars: 113
['ANO_CMPT', 'AUD_JUST', 'CAR_INT', 'CBOR', 'CEP', 'CGC_HOSP', 'CID_ASSO', 'CID_MORTE', 'CID_NOTIF', 'CNAER', 'CNES', 'CNPJ_MANT', 'COBRANCA', 'COD_IDADE', 'COMPLEX', 'CONTRACEP1', 'CONTRACEP2', 'CPF_AUT', 'DIAGSEC1', 'DIAGSEC2']


Filter the dictionary to 113 columns in the admissions dataset

In [6]:
#Keep only rows in dict_df where 'id' matches one of admissions dataset columns
filtered_dict = dict_df[dict_df['id'].isin(matching_vars)]
print(filtered_dict.shape)
filtered_dict.head()

(113, 3)


,id,type,description
0,UF_ZI,bigint,MunicÃ­pio gestor
1,ANO_CMPT,bigint,Ano de processamento da AIH
2,MES_CMPT,text,MÃªs de processamento da AIH
3,ESPEC,text,Especialidade do leito
4,CGC_HOSP,text,CNPJ do estabelecimento


## 2.Translate Portuguese descriptions into English

In [8]:
from deep_translator import GoogleTranslator

# Create translator once
translator = GoogleTranslator(source="pt", target="en")

def safe_translate(text):
    # Handle NaN, None, or non-string inputs
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # Handle "Sem descrição" gracefully
    if text.lower() in ["sem descrição", "sem descricao"]:
        return "No description"

    try:
        return translator.translate(text)
    except Exception:
        return text  # fallback: keep original

# Apply translation only to unique values
unique_texts = filtered_dict['description'].unique()
translations = {txt: safe_translate(txt) for txt in unique_texts}
filtered_dict['english_description'] = filtered_dict['description'].map(translations)

# Save once, reuse later
filtered_dict.to_csv("filtered_dict_translated.csv", index=False)


## 2.1 Generate a Markdown table for data_dictionary.md

In [10]:
def make_markdown_table(df):
    lines = []
    # Create the header row
    lines.append("| Column | Official Description | English Description | Category | Verified |")
    lines.append("| ------ | -------------------- | ------------------- | -------- | -------- |")
    # Create the data rows
    for _, row in df.iterrows():
        col = row['id']
        desc_pt = row['description']
        desc_en = row['english_description']
        lines.append(f"| {col} | {desc_pt} | {desc_en} |  |  |")
    return "\n".join(lines)

markdown_text = make_markdown_table(filtered_dict)  

print("\n".join(markdown_text.splitlines()[:20]))

| Column | Official Description | English Description | Category | Verified |
| ------ | -------------------- | ------------------- | -------- | -------- |
| UF_ZI | MunicÃ­pio gestor | Managing municipality |  |  |
| ANO_CMPT | Ano de processamento da AIH | AIH processing year |  |  |
| MES_CMPT | MÃªs de processamento da AIH | AIH Processing Month |  |  |
| ESPEC | Especialidade do leito | Bed specialty |  |  |
| CGC_HOSP | CNPJ do estabelecimento | CNPJ of the establishment |  |  |
| N_AIH | NÃºmero da AIH | AIH number |  |  |
| IDENT | IdentificaÃ§Ã£o do tipo da AIH | Identification of the AIH type |  |  |
| CEP | CEP do paciente | Patient zip code |  |  |
| MUNIC_RES | MunicÃ­pio de residÃªncia do paciente | Patient’s municipality of residence |  |  |
| NASC | Data de nascimento do paciente | Patient's date of birth |  |  |
| SEXO | Sexo do paciente | Patient's gender |  |  |
| UTI_MES_IN | Sem descriÃ§Ã£o | No description |  |  |
| UTI_MES_AN | Sem descriÃ§Ã£o | No description | 

Save the markdown_text  into data_dictionary.md

In [ ]:
with open("data_dictionary.md", "w", encoding="utf-8") as f:
    f.write(markdown_text)

print("Data dictionary saved to data_dictionary.md")

Open and Preview the Markdown File

In [12]:
#open the file with uft-8 encoding to check the content
with open("data_dictionary.md", "r", encoding="utf-8") as f:
    content = f.read()
#print first 30 lines to check formatting
print("\n".join(content.splitlines()[:30]))

| Column | Official Description | English Description | Category | Verified |
| ------ | -------------------- | ------------------- | -------- | -------- |
| UF_ZI | MunicÃ­pio gestor | Managing municipality |  |  |
| ANO_CMPT | Ano de processamento da AIH | AIH processing year |  |  |
| MES_CMPT | MÃªs de processamento da AIH | AIH Processing Month |  |  |
| ESPEC | Especialidade do leito | Bed specialty |  |  |
| CGC_HOSP | CNPJ do estabelecimento | CNPJ of the establishment |  |  |
| N_AIH | NÃºmero da AIH | AIH number |  |  |
| IDENT | IdentificaÃ§Ã£o do tipo da AIH | Identification of the AIH type |  |  |
| CEP | CEP do paciente | Patient zip code |  |  |
| MUNIC_RES | MunicÃ­pio de residÃªncia do paciente | Patient’s municipality of residence |  |  |
| NASC | Data de nascimento do paciente | Patient's date of birth |  |  |
| SEXO | Sexo do paciente | Patient's gender |  |  |
| UTI_MES_IN | Sem descriÃ§Ã£o | No description |  |  |
| UTI_MES_AN | Sem descriÃ§Ã£o | No description | 

In [13]:
row_count = len(content.splitlines())-2 #subtract 2 for header + separator
print(f"total rows in markdown file: {row_count}")

total rows in markdown file: 113


## 3. Category Mapping

In [14]:
#Domain1:Geography variables
geo_vars = [
    "UF_ZI",
    "CEP",
    "MUNIC_RES",
    "MUNIC_MOV"
]

filtered_dict.loc[filtered_dict['id'].isin(geo_vars), "Category"] = "Geography"

#Domain2:Hospital/Facility
facility_vars = [
    "ESPEC",
    "CGC_HOSP",
    "NATUREZA",
    "NAT_JUR",
    "GESTAO",
    "CNES",
    "CNPJ_MANT",
    "COMPLEX",
    "REGCT"
]
filtered_dict.loc[filtered_dict['id'].isin(facility_vars), "Category"] = "Hospital/Facility"

#Domain3:Patient
patient_vars = [
    "SEXO",
    "IDADE",
    "COD_IDADE",
    "NACIONAL",
    "HOMONIMO",
    "NUM_FILHOS",
    "INSTRU",
    "CONTRACEP1",
    "CONTRACEP2",
    "GESTRISCO",
    "CBOR",
    "VINCPREV",
    "RACA_COR",
    "ETNIA",
    "INSC_PN"
]

filtered_dict.loc[filtered_dict['id'].isin(patient_vars), "Category"] = "Patient"

#Domain4:Admission/Discharge
admission_vars = [
    "N_AIH",
    "IDENT",
    "DIAS_PERM",
    "SEQUENCIA",
    "REMESSA",
    "CAR_INT",
    "MORTE",
    "COBRANCA"
]
filtered_dict.loc[filtered_dict['id'].isin(admission_vars), "Category"] = "Admission/Discharge"

#Domain5:Time
time_vars = [
    "ANO_CMPT",
    "MES_CMPT",
    "NASC",
    "DT_INTER",
    "DT_SAIDA",
    "GESTOR_DT"
]

filtered_dict.loc[filtered_dict['id'].isin(time_vars), "Category"] = "Time"

#Domain6:Diagnosis
diagnosis_vars = [
    "DIAG_PRINC",
    "DIAG_SECUN",
    "DIAGSEC1",
    "DIAGSEC2",
    "DIAGSEC3",
    "DIAGSEC4",
    "DIAGSEC5",
    "DIAGSEC6",
    "DIAGSEC7",
    "DIAGSEC8",
    "DIAGSEC9",
    "TPDISEC1",
    "TPDISEC2",
    "TPDISEC3",
    "TPDISEC4",
    "TPDISEC5",
    "TPDISEC6",
    "TPDISEC7",
    "TPDISEC8",
    "TPDISEC9",
    "CID_ASSO",
    "CID_MORTE",
    "CID_NOTIF"
]
filtered_dict.loc[filtered_dict['id'].isin(diagnosis_vars), "Category"] = "Diagnosis"

#Domain7:Procedures
procedure_vars = [
    "PROC_SOLIC",
    "PROC_REA"
]
filtered_dict.loc[filtered_dict['id'].isin(procedure_vars), "Category"] = "Procedures"

#Domain8:Financial
financial_vars = [
    "VAL_SH",
    "VAL_SP",
    "VAL_SADT",
    "VAL_RN",
    "VAL_ACOMP",
    "VAL_ORTP",
    "VAL_SANGUE",
    "VAL_SADTSR",
    "VAL_TRANSP",
    "VAL_OBSANG",
    "VAL_PED1AC",
    "VAL_TOT",
    "VAL_UTI",
    "US_TOT",
    "FINANC",
    "FAEC_TP",
    "VAL_SH_FED",
    "VAL_SP_FED",
    "VAL_SH_GES",
    "VAL_SP_GES",
    "VAL_UCI",
    "RUBRICA",
    "TOT_PT_SP"
]
filtered_dict.loc[filtered_dict['id'].isin(financial_vars), "Category"] = "Financial"

#Domain9:Resource Utilization
utilization_vars = [
    "UTI_MES_IN",
    "UTI_MES_AN",
    "UTI_MES_AL",
    "UTI_MES_TO",
    "MARCA_UTI",
    "UTI_INT_IN",
    "UTI_INT_AN",
    "UTI_INT_AL",
    "UTI_INT_TO",
    "MARCA_UCI",
    "DIAR_ACOM",
    "QT_DIARIAS"
]
filtered_dict.loc[filtered_dict['id'].isin(utilization_vars), "Category"] = "Resource Utilization"

#Domain10:Aministrative
admin_vars = [
    "GESTOR_COD",
    "GESTOR_CPF",
    "GESTOR_TP",
    "AUD_JUST",
    "SIS_JUST",
    "CPF_AUT",
    "SEQ_AIH5",
    "CNAER",
    "NUM_PROC"

]
filtered_dict.loc[filtered_dict['id'].isin(admin_vars), "Category"] = "Administrative"

#Domain11:Clinical
clinical_vars = [
    "INFEHOSP",
    "IND_VDRL"
]

filtered_dict.loc[filtered_dict["id"].isin(clinical_vars),"Category"] = "Clinical"

In [15]:
def update_dictionary_file (df, filename ="data_dictionary.md"):
    """update the markdown file with the current dataframe content"""
    markdown_text = df.to_markdown(index=False)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(markdown_text)
    print(f"{filename} updated with latest categories.")

In [ ]:
update_dictionary_file(filtered_dict, filename="data_dictionary.md")

In [19]:
filtered_dict.to_csv("data_dict_translated_mapped.csv", index=False)

## 4. Data Validation
### 4.1 Time

In [ ]:
# Time domain variables
time_vars = [ "ANO_CMPT",
    "MES_CMPT",
    "NASC",
    "DT_INTER",
    "DT_SAIDA",
    "GESTOR_DT"]

time_df = df[time_vars]

print("Data types:\n", time_df.dtypes, "\n")
print("Missing values:\n", time_df.isna().sum(), "\n")
print("Sample values:\n", time_df.head(),"\n")
print("Value ranges:\n")
for col in time_vars: 
    print(f"{col}: min={time_df[col].min()}, max={time_df[col].max()}")
    

#### Findings: Time Variables
- ANO_CMPT: Year of processing, fixed at 2024.
- MES_CMPT: Month of processing, fixed at January.
- NASC: Patient birthdate, encoded as YYYYMMDD. Range 1895–2024.
- DT_INTER: Admission date, encoded as YYYYMMDD. Range 2008–2024.
- DT_SAIDA: Discharge date, encoded as YYYYMMDD. Range Aug 2023–Jan 2024.
- GESTOR_DT: Mostly missing (≈1.1M nulls).

In [ ]:
df["NASC_dt"]= pd.to_datetime(df["NASC"].astype(str),format="%Y%m%d")
df["DT_INTER_dt"]= pd.to_datetime(df["DT_INTER"].astype(str),format="%Y%m%d")
df["DT_SAIDA_dt"]= pd.to_datetime(df["DT_SAIDA"].astype(str),format="%Y%m%d")   

In [ ]:
(df["DT_INTER_dt"] > df["DT_SAIDA_dt"]).sum()

## Validation Check: Admission vs. Discharge

- Rule: Admission date must be on or before discharge date.
- Result: 0 violations.
- Interpretation: All records satisfy the logical consistency rule.  
  No cases found where admission occurs after discharge.

In [ ]:
(df["NASC_dt"] > df["DT_INTER_dt"]).sum()

## Validation Check: Birthdate vs. Admission

- Rule: Birthdate must be on or before admission date.
- Result: 0 violations.
- Interpretation: All records satisfy this rule; no impossible timelines detected.

In [ ]:
#to validate whether NASC values are in future realative to dataset's processing period (2024)
(df["NASC_dt"] > pd.Timestamp("2024-01-31")).sum()

## Validation Check: Birth Date vs. Dataset Reference Date

- Rule: Birth date must not be later than 2024-01-31.
- Result: 0 violations.
- Interpretation: No future birth dates detected.

In [ ]:
((df["DT_INTER_dt"]- df["NASC_dt"]).dt.days/365.25 > 120).sum()

In [ ]:
df.loc[
    ((df["DT_INTER_dt"]-df["NASC_dt"]).dt.days/365.25 > 120),
    ["NASC_dt", "DT_INTER_dt", "DT_SAIDA_dt", "N_AIH", "CNES"]
]

In [ ]:
flagged = df.loc[
    ((df["DT_INTER_dt"] - df["NASC_dt"]).dt.days/365.25 > 120),
    ["NASC_dt", "DT_INTER_dt", "IDADE","COD_IDADE", "N_AIH", "CNES"]
]

print(flagged.to_string(index=False))

## Validation Check: Birth Date vs. Recorded Age

Rule: Birth date should be consistent with the recorded patient age.
Result: 11 records have birth dates implying an age >120 years.
Observation: All 11 records have `COD_IDADE = 5` (years) and recorded `IDADE` values between 23 and 28.
Interpretation: The anomalous `NASC` values are inconsistent with the recorded age fields and may represent placeholder/default birth dates.
Action: Retain the original values; do not alter or impute them. Flag for source-level investigation.

In [ ]:
#Admissions dating back to 2008 (DT_INTER) in a January 2024 dataset
df.loc[df["DT_INTER_dt"].dt.year < 2023,
       ["DT_INTER_dt", "DT_SAIDA_dt", "N_AIH", "CNES"]]

In [ ]:
df.loc[
    df["DT_INTER_dt"].dt.year < 2023,
    "DT_INTER_dt"
].dt.year.value_counts().sort_index()

In [ ]:
((df["DT_SAIDA_dt"]-df["DT_INTER_dt"]).dt.days != df["DIAS_PERM"]).sum()

#### Validation Check: Length of Stay

Result: 10,887 records where `DIAS_PERM` does not equal the calendar-day difference between admission and discharge.
Interpretation: SIH/SUS-specific conventions may explain the discrepancy (e.g., billing rules, administrative closures).
Action: Retain `DIAS_PERM` as reported; do not recalculate. Flag for source-level clarification.

### 4.2 Geography validation

In [ ]:
geo_vars = [
    "UF_ZI",
    "CEP",
    "MUNIC_RES",
    "MUNIC_MOV"
]

geo_df = df[geo_vars]
print("Data types:\n", geo_df.dtypes, "\n")
print("Missing values:\n", geo_df.isna().sum(), "\n")
print("Sample values:\n", geo_df.head(),"\n")
print("Value ranges:\n")
for col in geo_vars: 
    print(f"{col}: min={geo_df[col].min()}, max={geo_df[col].max()}")

In [ ]:
(df["CEP"].astype(str).str.len() !=8).sum()

In [ ]:
df.loc[
    df["CEP"].astype(str).str.len() !=8,
    "CEP"
].value_counts().head(20)

In [ ]:
geo_df[["UF_ZI", "MUNIC_RES", "MUNIC_MOV"]].nunique()

### 4.2 Summary
## Geography profiling is clean structurally
Data types: all four variables are int64.
Missing values: none — every record has a value.
`UF_ZI`:range: 110000–530000 
`CEP`:range: 1001000–99990000 
`MUNIC_RES`:range: 110001–530010 
`MUNIC_MOV`:range: 110001–530010 

## Validation Check: CEP Format
Result: 91,461 records have fewer than 8 digits when stored as integers.
Observation: Sample values such as `7082050` and `6730000` are consistent with CEPs that may have leading zeros removed.
Interpretation: The finding appears to be primarily a storage/formatting issue rather than evidence of invalid CEP values.
Action: Retain the original `CEP` field. For analysis or presentation requiring CEP formatting, treat it as an 8-character string with leading zeros restored.

## Cardinality
UF_ZI → 2,134 unique values
MUNIC_RES → 5,570 unique values
MUNIC_MOV → 3,009 unique values
Interpretation:The three geographic code fields show substantial variation and are not fixed or near-constant across the dataset.

### 4.3 Hospital/Facility domain validation

In [ ]:
facility_vars = [
    "ESPEC",
    "CGC_HOSP",
    "NATUREZA",
    "NAT_JUR",
    "GESTAO",
    "CNES",
    "CNPJ_MANT",
    "COMPLEX",
    "REGCT"
]

print("Data types:\n", df[facility_vars].dtypes, "\n")
print("Missing values:\n", df[facility_vars].isna().sum(), "\n")
print("Sample values:\n", df[facility_vars].head(), "\n")
print("Cardinality:\n", df[facility_vars].nunique(), "\n")

print("value ranges:\n")
for col in facility_vars:
    if df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

In [ ]:
df["CNES"].value_counts().head(10)

## 4.3 Summary
 `ESPEC`    : 15 unique; range 1–87    
 `CGC_HOSP` : 282,035 missing; stored as float
 `NATUREZA` : **1 unique value; all = 0**     
 `NAT_JUR`  : 27 unique                       
 `GESTAO`   : 2 unique                        
 `CNES`     : 4,513 unique                   
 `CNPJ_MANT`: 532,289 missing; stored as float 
 `COMPLEX`  : 2 unique                        
 `REGCT`    :9 unique                        

## Validation Check: CNES Distribution
 Interpretation: `CNES` shows substantial facility-level variation with 4,513 unique facility codes.

## 4.4 Patient Domain Validation

In [ ]:
patient_vars = [
    "SEXO",
    "IDADE",
    "COD_IDADE",
    "NACIONAL",
    "HOMONIMO",
    "NUM_FILHOS",
    "INSTRU",
    "CONTRACEP1",
    "CONTRACEP2",
    "GESTRISCO",
    "CBOR",
    "VINCPREV",
    "RACA_COR",
    "ETNIA",
    "INSC_PN"
]

patient_df = df[patient_vars]

print("Data types:\n", df[patient_vars].dtypes, "\n")
print("Missing values:\n", df[patient_vars].isna().sum(), "\n")
print("Sample values:\n", df[patient_vars].head(), "\n")
print("Cardinality:\n", df[patient_vars].nunique(), "\n")

print("Value ranges:\n")
for col in patient_vars:
    if df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")


In [ ]:
patient_df["SEXO"].value_counts()

In [ ]:
print("Unique RACA_COR values:\n", df["RACA_COR"].unique())
print("Counts:\n", df["RACA_COR"].value_counts())

## Patient domain summary
Patient profiling is structurally clean: no missing values across all 15 variables.
IDADE: 0–99, 100 unique → plausible range.
COD_IDADE: 4 unique, 2–5 → categorical code; needs no range interpretation beyond that.
SEXO: 2 unique values (codes 1 and 3); no missing values. No unexpected third code observed in the dataset.
NACIONAL → 137 unique nationality codes.
NUM_FILHOS → 0–40. Upper bound is high but still plausible.
INSTRU → 5 unique values (education levels).
CONTRACEP1/2 → 14 unique codes each.
GESTRISCO → binary (0/1).
RACA_COR → has 6 observed codes: 1–5 and 99. Code 99 occurs 282 times and should be treated as an unknown/not-declared category for analysis, subject to confirmation from the SIH/SUS coding specification.
ETNIA → 201 unique values, range 0–263.
INSC_PN → very high cardinality (18,455 unique), range up to 999,999,999,999 → program enrollment IDs.likely an identifier/registration field rather than an analytical numeric measure.
CBOR: range 0–515105 and 5 unique values is interesting; likely coded categories despite the numeric-looking range.

# Validation Check: RACA_COR Coding
- Rule: Codes 1–6 represent the substantive race/color categories; additional codes should be reviewed against the SIH/SUS coding specification.
- Result: Found codes [1, 2, 3, 4, 5, 99]. Code 99 appears in 282 records.
- Observation: Code 99 falls outside the substantive 1–6 categories.
- Interpretation: Code 99 may represent unknown/not declared information; this should be confirmed against the SIH/SUS specification before recoding.
- Action: Retain the original value. For analysis, consider grouping 99 separately only after confirming its official meaning.

## 4.5 Admission/Discharge Domain

In [ ]:
admission_vars = [
    "N_AIH",
    "IDENT",
    "DIAS_PERM",
    "SEQUENCIA",
    "REMESSA",
    "CAR_INT",
    "MORTE",
    "COBRANCA"
]
admission_df = df[admission_vars]

print("Data types:\n", df[admission_vars].dtypes, "\n")
print("Missing values:\n", df[admission_vars].isna().sum(), "\n")
print("Sample values:\n", df[admission_vars].head(), "\n")
print("Cardinality:\n", df[admission_vars].nunique(), "\n")

print("Value ranges:\n")
for col in admission_vars:
    if df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")


## Summary
Admission/Discharge profiling is structurally clean: no missing values in any of the 8 variables.
# Key findings:
N_AIH → 1,107,309 unique values out of 1,107,877 records; nearly unique identifier.
IDENT →  2 unique values; observed values fall within the coded range 1–5; categorical.
DIAS_PERM → 0–358 days; 179 unique values.
SEQUENCIA → 102,282 unique values.
REMESSA → 2,134 unique values; string field, as expected.
CAR_INT → 5 unique values, range 1–6.
MORTE → binary 0/1.
COBRANCA → 28 unique codes, range 11–67.

# Validation Check: DIAS_PERM (Length of Stay)
- Rule: Hospital stays should be non-negative and within plausible clinical limits.
- Result: Range 0–358 days.
- Interpretation: Values are non-negative, with a maximum of 358 days. No immediately implausible values identified from range inspection.
- Action: Retain values as-is; note upper bound for context in analysis.

## 4.6 Diagnosis Domain Validation

In [ ]:
diagnosis_vars = [
    "DIAG_PRINC",
    "DIAG_SECUN",
    "DIAGSEC1",
    "DIAGSEC2",
    "DIAGSEC3",
    "DIAGSEC4",
    "DIAGSEC5",
    "DIAGSEC6",
    "DIAGSEC7",
    "DIAGSEC8",
    "DIAGSEC9",
    "TPDISEC1",
    "TPDISEC2",
    "TPDISEC3",
    "TPDISEC4",
    "TPDISEC5",
    "TPDISEC6",
    "TPDISEC7",
    "TPDISEC8",
    "TPDISEC9",
    "CID_ASSO",
    "CID_MORTE",
    "CID_NOTIF"
]

print("Data types:\n", df[diagnosis_vars].dtypes, "\n")
print("Missing values:\n", df[diagnosis_vars].isna().sum(), "\n")
print("Sample values:\n", df[diagnosis_vars].head(), "\n")
print("Cardinality:\n", df[diagnosis_vars].nunique(), "\n")

print("Value ranges:\n")
for col in diagnosis_vars:
    if df[col].dtype in ["int64","float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

## 4.6 Summary

Diagnosis profiling shows complete population of the primary diagnosis field but substantial sparsity across optional secondary diagnosis fields.
- DIAG_PRINC: 7,746 unique values; no missing values.
- DIAG_SECUN: constant 0.
- DIAGSEC1–7: progressively increasing missingness.
- DIAGSEC8–9: completely missing.
- TPDISEC1–7: 3 categorical values each (0–2).
- TPDISEC8–9: constant 0.
- CID_ASSO: constant 0.
- CID_MORTE: constant 0.
- CID_NOTIF: ~97.5% missing; 29 unique values.

# Data types
- Diagnosis code fields (`DIAG_PRINC`, `DIAGSEC1–7`) are stored as strings, which is appropriate for coded diagnosis values.
- `DIAGSEC8` and `DIAGSEC9` are `float64` because they are completely missing; this reflects pandas type inference for all-null columns.
- Categorical/indicator fields (`DIAG_SECUN`, `TPDISEC1–9`, `CID_ASSO`, `CID_MORTE`) are stored as integers.
- `CID_NOTIF` is stored as string.

# Interpretation
Primary diagnosis is consistently populated. Secondary diagnosis fields appear to represent optional/repeated diagnosis slots, with progressively fewer records containing values. Completely unused fields are retained because they are part of the source schema.

# Validation finding
No immediate structural anomaly identified in the diagnosis fields. ICD-code validity will not be independently validated at this stage because an official reference-code lookup has not been incorporated into the workflow.

## 4.7 Procedures Domain

In [ ]:
procedure_vars = [
    "PROC_SOLIC",
    "PROC_REA",
    "NUM_PROC",
]

print("Data types:\n", df[procedure_vars].dtypes, "\n")
print("Missing values:\n", df[procedure_vars].isna().sum(), "\n")
print("Sample values:\n", df[procedure_vars].head(), "\n")
print("Cardinality:\n", df[procedure_vars].nunique(), "\n")

print("Value ranges:\n")
for col in procedure_vars:
    if df[col].dtype in ["int64","float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

## Summary
- `PROC_SOLIC`: `int64`, no missing values, 1,648 unique procedure codes.
- `PROC_REA`: `int64`, no missing values, 1,597 unique procedure codes.
- `NUM_PROC`: `float64`, 100% missing and therefore unused in this dataset.

# Interpretation
-The requested and performed procedure fields are fully populated and show substantial code variation. Their numeric storage is appropriate for coded procedure identifiers.
-`NUM_PROC` is completely unpopulated; its `float64` dtype is a consequence of pandas representing an all-missing column and does not indicate a data-quality problem.
-No immediate structural anomaly was identified in the available procedure fields.

# Variable availability note
`CBOR_PROC` was included in the initial procedure-domain variable list but is not present in the loaded dataset. It was therefore excluded from dataset-level profiling and validation.

## 4.8 Financial Domain

In [ ]:
financial_vars = [
    "VAL_SH",
    "VAL_SP",
    "VAL_SADT",
    "VAL_RN",
    "VAL_ACOMP",
    "VAL_ORTP",
    "VAL_SANGUE",
    "VAL_SADTSR",
    "VAL_TRANSP",
    "VAL_OBSANG",
    "VAL_PED1AC",
    "VAL_TOT",
    "VAL_UTI",
    "US_TOT",
    "FINANC",
    "FAEC_TP",
    "VAL_SH_FED",
    "VAL_SP_FED",
    "VAL_SH_GES",
    "VAL_SP_GES",
    "VAL_UCI",
    "RUBRICA",
    "TOT_PT_SP"
]

print("Data types:\n", df[financial_vars].dtypes, "\n")
print("Missing values:\n", df[financial_vars].isna().sum(), "\n")
print("Sample values:\n", df[financial_vars].head(), "\n")
print("Cardinality:\n", df[financial_vars].nunique(), "\n")

print("Value ranges:\n")
for col in financial_vars:
    if df[col].dtype in ["int64","float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

In [ ]:
financial_value_vars = [
    col for col in financial_vars
    if col.startswith("VAL_") or col == "US_TOT"
]

(df[financial_value_vars] < 0).sum().sum()

## Summary
The Financial domain contains **23 variables** covering hospital payment components, reimbursement amounts, funding sources, and related financial indicators.

# Data types: Most financial amount fields (`VAL_*`, `US_TOT`, `FAEC_TP`) are stored as `float64`, appropriate for monetary/amount fields. `FINANC`, `RUBRICA`, and `TOT_PT_SP` are stored as `int64`, representing coded/indicator fields.

# Missing values: 
-All financial amount fields are fully populated.
-`FAEC_TP` has **1,039,881 missing values (~93.9%)**, indicating that this field is sparsely populated in this dataset.
- No other financial variable has missing values.

# Cardinality and usage:
-Several major financial fields show substantial variation, including `VAL_SH`, `VAL_SP`, `VAL_TOT`, `VAL_UTI`, and `US_TOT`.
-Several fields are **constant zero** across all records: `VAL_SADT`, `VAL_RN`, `VAL_ACOMP`, `VAL_ORTP`, `VAL_SANGUE`, `VAL_SADTSR`, `VAL_TRANSP`, `VAL_OBSANG`, `VAL_PED1AC`, `RUBRICA`, and `TOT_PT_SP`.
-These constant fields appear **unused in this dataset**, rather than representing data-quality errors.

# Validation: Negative financial values: 0 negative values across the financial amount fields.
-All observed financial amounts have a minimum of `0`.
-This indicates no obvious sign-related anomaly in the financial data.

# Overall interpretation: The Financial domain is structurally clean. Major financial fields are populated and show substantial variation, while several source fields are unused/constant at zero. `FAEC_TP` is highly sparse. No negative financial amounts were identified.

## 4.9:Resource Utilization Domain

In [ ]:
utilization_vars = [
    "UTI_MES_IN",
    "UTI_MES_AN",
    "UTI_MES_AL",
    "UTI_MES_TO",
    "MARCA_UTI",
    "UTI_INT_IN",
    "UTI_INT_AN",
    "UTI_INT_AL",
    "UTI_INT_TO",
    "MARCA_UCI",
    "DIAR_ACOM",
    "QT_DIARIAS",
]

print("Data types:\n", df[utilization_vars].dtypes, "\n")
print("Missing values:\n", df[utilization_vars].isna().sum(), "\n")
print("Sample values:\n", df[utilization_vars].head(), "\n")
print("Cardinality:\n", df[utilization_vars].nunique(), "\n")

print("Value ranges:\n")
for col in utilization_vars:
    if df[col].dtype in ["int64","float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

In [ ]:
resource_numeric_vars = [
    "UTI_MES_IN", "UTI_MES_AN", "UTI_MES_AL", "UTI_MES_TO","MARCA_UTI",
    "UTI_INT_IN", "UTI_INT_AN", "UTI_INT_AL", "UTI_INT_TO",
    "DIAR_ACOM", "QT_DIARIAS"
]

(df[resource_numeric_vars] < 0).sum().sum()

## Summary
- All 11 variables are stored as `int64`.
- No missing values were identified.
- `UTI_MES_TO` and `UTI_INT_TO` show substantial variation, while the corresponding `IN`, `AN`, and `AL` fields are constant at zero in this dataset.
- `MARCA_UCI`, `DIAR_ACOM`, and `QT_DIARIAS` show meaningful variation.
- All observed utilization/day quantities are non-negative.
- Several fields are constant zero and appear unused in this dataset extract.
- The source dictionary provides no descriptions for several `UTI_*` variables; therefore, their specific `IN/AN/AL` semantics are not inferred at this stage.

Negative-value check: 0 negative values were identified across the applicable utilization/day variables.

# Validation finding
No structural or negative-value anomaly was identified in the Resource Utilization domain.

## 4.10 Administrative Domain

In [ ]:
admin_vars = [
    "GESTOR_COD",
    "GESTOR_CPF",
    "GESTOR_TP",
    "AUD_JUST",
    "SIS_JUST",
    "CPF_AUT",
    "SEQ_AIH5",
     "CNAER"    

]

print("Data types:\n", df[admin_vars].dtypes, "\n")
print("Missing values:\n", df[admin_vars].isna().sum(), "\n")
print("Sample values:\n", df[admin_vars].head(), "\n")
print("Cardinality:\n", df[admin_vars].nunique(), "\n")

print("Value ranges:\n")
for col in admin_vars:
    if df[col].dtype in ["int64","float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

In [ ]:
admin_numeric_vars = [
    "GESTOR_COD",
    "GESTOR_TP",
    "SEQ_AIH5",
     "CNAER"
]

(df[admin_numeric_vars] < 0).sum().sum()

## summary
- 8 variables profiled.
- No missing values in the core administrative coded fields.
- `AUD_JUST` and `SIS_JUST` are highly sparse.
- `CPF_AUT` is completely missing in this dataset extract.
- `GESTOR_CPF` is an identifier-like field and was not interpreted as a quantitative measure.
- `SEQ_AIH5` has low cardinality and is treated as a coded/sequence field.
- No negative values were identified in the applicable numeric administrative fields.

## Validation finding
No structural or negative-value anomaly was identified in the Administrative/Audit domain.

## 4.11 Clinical Domain

In [ ]:
clinical_vars = [
    "INFEHOSP",
    "IND_VDRL"
]


print("Data types:\n", df[clinical_vars].dtypes, "\n")
print("Missing values:\n", df[clinical_vars].isna().sum(), "\n")
print("Sample values:\n", df[clinical_vars].head(), "\n")
print("Cardinality:\n", df[clinical_vars].nunique(), "\n")

print("Value ranges:\n")
for col in clinical_vars:
    if df[col].dtype in ["int64","float64"]:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"{col}: non-numeric (skip range)")

In [ ]:
(~df["IND_VDRL"].isin([0, 1])).sum()

## Summary
- `INFEHOSP` is completely missing (1,107,877 records) and has no observed values.
- `IND_VDRL` is complete and contains two values (0 and 1).
- No range validation is applicable to `INFEHOSP` because the field is entirely unpopulated.
- `IND_VDRL` requires only a binary coding check.

## Validation finding
`IND_VDRL` contains only valid binary values (0/1).
`INFEHOSP` is retained unchanged and documented as completely unpopulated in this dataset extract.